# Step 1. Company pick at random
Question: Which mid-size WC company do I build test exhibits from at random? 

Criteria (known cells only, so the pick uses no hindsight):
- **Complete (all 10 AYs):** full triangle, so every maturity gets graded and every AY feeds the factor averages.
- **Size between p25 and p75 of median net EP per AY:** small lines' factors are 2–3.5× noisier (probe 8). The middle half is typical of the sample, so it's not dominated by noise or by one large book.
- **Ceded swing ≤ 0.5:** a changing reinsurance program can change net development (excess of loss speeds it up). That would add a chain ladder error that has nothing to do with the method (probe 7).
- **No negative case cells (case ≤ −2):** reported below paid distorts the reported triangle's factors (probe 3). A case of −1 is $000 rounding, so it's allowed.

Random pick among eligible lines, seed fixed before running: I can't steer it toward a company that flatters a method, and the seed makes the pick reproducible.

In [5]:
# Phase 2, step 1: random pick of one mid-size WC company (known cells only)
import numpy as np
import pandas as pd

SEED = 20261102  # fixed before running; run once, never re-roll

wc_clean = pd.read_csv("../data/clean/wc_clean.csv")

df = wc_clean.copy()  # 60 clean WC lines
known = df[df["AccidentYear"] + df["DevelopmentLag"] - 1 <= 2007].copy()
known["case"] = known["IncurredLosses"] - known["BulkLoss"] - known["CumPaidLoss"]
prem = known[known["DevelopmentLag"] == 1]  # one row per company-AY

ay = prem.groupby("GRCODE")
lines = pd.DataFrame({
    "GRNAME": ay["GRNAME"].first(),
    "n_ay": ay["AccidentYear"].nunique(),
    "size": ay["EarnedPremNet"].median(),  # $000, median net EP per AY
    "swing": (prem["EarnedPremCeded"] / prem["EarnedPremDIR"])
             .groupby(prem["GRCODE"]).agg(lambda s: s.max() - s.min()),
    "neg_case": (known["case"] <= -2).groupby(known["GRCODE"]).sum(),
})
assert len(lines) == 60 and (lines["n_ay"] == 10).sum() == 46  # 60 − 14 incomplete

p25, p75 = lines["size"].quantile([0.25, 0.75])
ok = (
    (lines["n_ay"] == 10)
    & lines["size"].between(p25, p75)
    & (lines["swing"] <= 0.5)
    & (lines["neg_case"] == 0)
)
eligible = lines[ok].sort_index()  # sorted, so the pick doesn't depend on row order
print(f"Size p25–p75: {p25:,.0f} to {p75:,.0f} ($000)")
print(f"Eligible: {len(eligible)} of {len(lines)}")
print(eligible.round(2))
assert len(eligible) >= 3, "Too few; decide in chat which criterion to loosen"

pick = np.random.default_rng(SEED).choice(eligible.index.to_numpy())
print(f"\nPick (seed {SEED}): GRCODE {pick}, {lines.loc[pick, 'GRNAME']}")

Size p25–p75: 1,464 to 29,021 ($000)
Eligible: 12 of 60
                                     GRNAME  n_ay     size  swing  neg_case
GRCODE                                                                     
671                   Farm Bureau Of MI Grp    10  20045.5   0.34         0
5940           American Contractors Ins Grp    10  28212.5   0.14         0
11126   Yasuda Fire & Marine Ins Co Of Amer    10  27411.5   0.34         0
13501                   Brethren Mut Ins Co    10   3792.0   0.44         0
14257                        IMT Ins Co Mut    10   5811.0   0.19         0
16446          Farmers Ins Co of Flemington    10   2174.5   0.18         0
18309              Pioneer State Mut Ins Co    10   1483.5   0.46         0
21172                       Vanliner Ins Co    10  17916.0   0.50         0
23663              National American Ins Co    10  16426.0   0.46         0
26433                     Harco Natl Ins Co    10   3652.5   0.34         0
40126                        Umi

# Step 3. Export the company

Splits GRCODE 23663's data so the Excel work only sees what was known at 12/31/2007?

- **Known workbook** (`excel/wc_23663.xlsx`): paid, reported and premium triangles for cells with AY + lag − 1 ≤ 2007 (55 per triangle). Steps 4–6 are built in this file.
- **Hidden answer key** (`data/hidden/wc_23663_hidden.csv`): the 45 cells after 2007. Not opened until step 10. Its SHA-256 hash is printed below, so at step 10 I can show the file wasn't changed.
- Values in $000. Paid = CumPaidLoss; reported = IncurredLosses − BulkLoss; premium = EarnedPremNet, one value per AY.

In [6]:
# Phase 2, step 3: export GRCODE 23663 known rows to Excel and hidden rows to a closed answer key
import hashlib
from pathlib import Path

GRCODE = 23663
VAL_YEAR = 2007  # valuation date 12/31/2007

co = wc_clean[wc_clean["GRCODE"] == GRCODE].copy()
assert co["GRNAME"].iloc[0].startswith("National American"), co["GRNAME"].iloc[0]
assert len(co) == 100  # complete line: 10 AYs x 10 lags

co["Reported"] = co["IncurredLosses"] - co["BulkLoss"]
co["CalYear"] = co["AccidentYear"] + co["DevelopmentLag"] - 1
is_known = co["CalYear"] <= VAL_YEAR

# Premium is one value per AY. Check it repeats on every row of the AY (reads premium only, no losses)
assert (co.groupby("AccidentYear")["EarnedPremNet"].nunique() == 1).all(), "Premium varies by lag; stop"

known, hidden = co[is_known], co[~is_known]
assert len(known) == 55 and len(hidden) == 45
assert known["CalYear"].max() == VAL_YEAR and hidden["CalYear"].min() == VAL_YEAR + 1

cols = ["AccidentYear", "DevelopmentLag", "CumPaidLoss", "IncurredLosses", "BulkLoss", "EarnedPremNet"]

info = pd.DataFrame([
    ("Company", f"{co['GRNAME'].iloc[0]} (GRCODE {GRCODE})"),
    ("Line", "Workers' compensation, net of reinsurance"),
    ("Source", "CAS Loss Reserving Database, AY 1998-2007 (cleaned: data/clean/wc_clean.csv)"),
    ("Valuation", "12/31/2007; known rows only (AY + lag - 1 <= 2007)"),
    ("Units", "$000"),
    ("Paid", "CumPaidLoss"),
    ("Reported", "IncurredLosses - BulkLoss (paid + case)"),
    ("Premium", "EarnedPremNet, one value per AY"),
], columns=["Item", "Value"])

xlsx = Path("../excel/wc_23663.xlsx")
xlsx.parent.mkdir(exist_ok=True)
assert not xlsx.exists(), f"{xlsx} exists; won't overwrite Excel work"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    info.to_excel(xw, sheet_name="Info", index=False)
    known[cols].sort_values(cols[:2]).to_excel(xw, sheet_name="Data", index=False)

key = Path("../data/hidden/wc_23663_hidden.csv")
key.parent.mkdir(parents=True, exist_ok=True)
hidden[["GRCODE", "AccidentYear", "DevelopmentLag", "CumPaidLoss", "Reported"]].to_csv(key, index=False)
sha = hashlib.sha256(key.read_bytes()).hexdigest()

del co, hidden  # nothing below this cell can show hidden values by accident
print(f"Known:  {len(known)} rows -> {xlsx}")
print(f"Hidden: 45 rows -> {key}")
print(f"Hidden SHA-256: {sha}")

Known:  55 rows -> ../excel/wc_23663.xlsx
Hidden: 45 rows -> ../data/hidden/wc_23663_hidden.csv
Hidden SHA-256: cc9c2b0a48ef282d0a932c3ba7d83536988eaca0ba8e840e2b33dfe9be9881aa


Export: 55 known rows -> excel/wc_23663.xlsx (Info, Data); 45 hidden rows -> data/hidden/wc_23663_hidden.csv, SHA-256 cc9c2b0a48ef282d…, not opened. Premium is the same at every lag for each AY, so lag 1 premium is the AY's premium.